In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
import os
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder


In [2]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [3]:

train_df = pd.read_csv("train (4).csv")
train_df


,ID,text,sentiment
0,21098,".с.,и спросил его: о Посланник Аллаха!Ты пори...",1
1,21099,Роднее всех родных Попала я в ГКБ №8 еще в дек...,1
2,21100,Непорядочное отношение к своим работникам Рабо...,2
3,21101,"). Отсутствуют нормативы, Госты и прочее, что ...",1
4,21102,У меня машина в руках 5 лет и это п...,1
...,...,...,...
189886,210984,"Мой юбилей я отмечал в ресторане "" Астория "" ....",2
189887,210985,"Отлично встретили, разместили в роскошном номе...",1
189888,210986,Была в Васаби на ст. метро Сенная . Во первых...,0
189889,210987,Ребята не стоит смотреть этот фильм. Вы молоды...,0


In [4]:
from sentence_transformers import SentenceTransformer

# loads model with mea pooling
MODEL_ID = "sergeyzh/BERTA"


In [5]:
y = train_df["sentiment"]
le = LabelEncoder()
le.fit(y)

LabelEncoder()

In [6]:


y = le.transform(y)

train_df = train_df.rename(columns={'sentiment': 'labels'})



train_dataset, eval_dataset, train_y, eval_y = train_test_split(
    train_df, y, test_size=0.2, random_state=42
)

# Reset the index of the train and eval datasets
train_dataset = train_dataset.reset_index(drop=True)
eval_dataset = eval_dataset.reset_index(drop=True)


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=3  # например, positive / neutral / negative
)



train_dataset = Dataset.from_pandas(train_dataset)
eval_dataset = Dataset.from_pandas(eval_dataset)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sergeyzh/BERTA and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/151912 [00:00<?, ? examples/s]

Map:   0%|          | 0/37979 [00:00<?, ? examples/s]

In [7]:
tokenized_train_dataset[0]

{'ID': 103731,
 'text': 'Отель расположен очень удачно(на мой взгляд)-в 10 минутах ходьбы от площади Султан Ахмет и в 10 минут от берега моря!!У меня был очень уютный номер с видом на море!!Стиль отеля очень сильно располагает к романтическому настроению!!Есть терраса,где можно полюбоваться на Мраморное море и ...на Голубую мечеть!!Захватывающий вид!!В номерах очень чисто.Уборка каждый день .Есть чайно-кофейный набор.Завтраки пристойные.Приветливый(русскоговорящий) персонал!!Одно "но"-близость мечетей не давала по ночам нормально выспаться(призыв к молитве был очень громкий через громкоговорители)!!А ещё мой отдых совпал с Рамаданом...Но это мелочи(можно выспаться дома...). Я рекомендовала бы этот отель!!',
 'labels': 1,
 'input_ids': [2,
  34772,
  18878,
  1850,
  43312,
  12,
  527,
  6708,
  14582,
  13,
  17,
  340,
  561,
  41547,
  46351,
  650,
  15508,
  44147,
  39828,
  346,
  340,
  561,
  4049,
  650,
  35126,
  13812,
  5,
  5,
  325,
  2827,
  2162,
  1850,
  45558,
  14


training_args = TrainingArguments(
    output_dir="./",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
                eval_strategy='steps',
            save_strategy='steps',
    report_to="none",  # <---- добавь эту строку
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,  # твои данные
    eval_dataset=tokenized_eval_dataset,
)

trainer.train()

In [8]:
import torch
torch.cuda.empty_cache()

In [9]:
from transformers import AutoModelForSequenceClassification

MODEL_ID="./checkpoint-34500"

#tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=3  # например, positive / neutral / negative
)

tokenizer = AutoTokenizer.from_pretrained( "sergeyzh/BERTA")



In [10]:
test_df = pd.read_csv("test (4).csv")
test_df

,ID,text
0,0,Развода на деньги нет Наблюдаюсь в Лайфклиник ...
1,1,Отель выбрали потому что рядом со стадионом. О...
2,2,"Вылечили Гноился с рождения глазик, в поликлин..."
3,3,Хорошее расположение.С вокзала дошли пешком.Но...
4,4,"Отличное месторасположение,прекрасный вид,особ..."
...,...,...
21093,21093,Несколько лет назад муж останавливался в этом ...
21094,21094,Спасли от боли После родов у меня появились бо...
21095,21095,з ролика понятно одно - девушка- наблюдатель ...
21096,21096,"Остались всем довольны, дружелюбный персонал, ..."


In [11]:
def tokenize_function_n(example):
    return tokenizer(example, padding="max_length", truncation=True)

In [15]:
# Make predictions on the test set
#test_dataset = Dataset.from_pandas(test_df)
#tokenized_test_dataset = tokenize_function(test_df['text'].tolist() ) #test_dataset.map(tokenize_function, batched=True)

#inputs = tokenize_function_n(test_df['text'].tolist()) #list(map(tokenize_function_n,test_df['text'].tolist()))
#inputs = tokenizer(test_df['text'].tolist() , return_tensors="pt", padding="max_length", truncation=True) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alltest = test_df['text'].tolist()
model.to(device)

predictions = []
model.eval()


batch_size = 32 

batches = [
    alltest[i:i + batch_size] 
    for i in range(0, len(alltest), batch_size)
]

with torch.no_grad():  # Отключаем вычисление градиентов
    for batch_texts in batches:

        inputs = tokenizer(batch_texts,                 padding="max_length",
                truncation=True,
                return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        outputs = model(**inputs)
        
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        # Сохранение результатов
        predictions.extend(probs.cpu().numpy())
    


In [19]:
predictions=np.array(predictions)

In [20]:
predictions[499:502]

array([[7.8042310e-01, 2.1866485e-01, 9.1199065e-04],
       [4.9821537e-02, 9.4961399e-01, 5.6442124e-04],
       [3.8739938e-02, 9.6064681e-01, 6.1328046e-04]], dtype=float32)

In [21]:
#predictions = trainer.predict(tokenized_test_dataset)

# Extract predicted labels
predicted_labels = predictions.argmax(axis=1)
predicted_labels = le.inverse_transform(predicted_labels)
print(predicted_labels)

[1 0 1 ... 0 1 1]


In [22]:
submission = pd.DataFrame({
    'ID': test_df['ID'].values,
    'target': predicted_labels
})
display(submission.head())
submission.to_csv('submission.csv', index=False)
print("submission.csv сохранён.")

,ID,target
0,0,1
1,1,0
2,2,1
3,3,0
4,4,1


submission.csv сохранён.
